<a href="https://colab.research.google.com/github/floriancomptejeu1707-creator/test/blob/main/ScrappingSignature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q -U google-generativeai requests beautifulsoup4

import requests
from bs4 import BeautifulSoup
import google.generativeai as genai
import sqlite3
import hashlib
import smtplib
from email.message import EmailMessage
from google.colab import userdata
import csv
import json

# --- 1. INITIALISATION DE LA BASE DE DONNÉES ---
conn = sqlite3.connect('memoire_mairies.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS historique (
        commune TEXT PRIMARY KEY,
        empreinte_texte TEXT
    )
''')
conn.commit()

def creer_empreinte(texte):
    """Génère un identifiant unique (Hash SHA-256) à partir du texte."""
    return hashlib.sha256(texte.encode('utf-8')).hexdigest()

# --- CONFIGURATION IA ---
CLE_API = userdata.get('API_KEY')
genai.configure(api_key=CLE_API)
model = genai.GenerativeModel('gemini-3.5-flash')

mairies = {}
nom_fichier_csv = 'mairies_nord_sites.csv'  # Remplace par ton vrai nom de fichier

try:
    with open(nom_fichier_csv, mode='r', encoding='utf-8') as fichier:
        lecteur = csv.DictReader(fichier)
        for ligne in lecteur:
            ville = ligne.get('ville', '').strip()
            site_brut = ligne.get('site_web', '').strip()

            url = ""
            if site_brut:
                # Cas 1 : Le format data.gouv avec du JSON genre [{"valeur": "..."}]
                if site_brut.startswith('['):
                    try:
                        donnees_json = json.loads(site_brut.replace("'", '"')) # Sécurité pour les guillemets simples/doubles
                        if donnees_json and isinstance(donnees_json, list):
                            url = donnees_json[0].get('valeur', '')
                    except:
                        pass
                # Cas 2 : C'est déjà une URL classique
                elif site_brut.startswith('http'):
                    url = site_brut

            # On enregistre si on a une ville et un site valide
            if ville and url and url.startswith('http'):
                mairies[ville] = url

    print(f"✅ {len(mairies)} mairies chargées avec succès et URLs nettoyées !")
except Exception as e:
    print(f"❌ Erreur lors de la lecture du fichier CSV : {e}")

mots_cles = ["voirie", "sécurité routière", "vitesse", "ralentisseur", "aménagement", "mobilier urbain", "banc", "potelet", "signalisation", "marquage", "stationnement", "travaux", "piéton", "zone 30", "panneau"]
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36..."}

# --- VARIABLES GLOBALES POUR L'E-MAIL STRUCTURÉ ---
opportunites_structurees = []
opportunites_trouvees = 0

for nom, url in mairies.items():
    print(f"\n{'='*50}")
    print(f"🌍 Analyse en cours pour : {nom}")

    try:
        reponse = requests.get(url, headers=headers, timeout=10)
        reponse.raise_for_status()
        soup = BeautifulSoup(reponse.text, 'html.parser')
        for script in soup(["script", "style"]):
            script.decompose()

        texte_brut = soup.get_text(separator='\n', strip=True)[:15000]

        # --- 2. LE SYSTÈME ANTI-DOUBLON (HASH) ---
        empreinte_actuelle = creer_empreinte(texte_brut)

        cursor.execute('SELECT empreinte_texte FROM historique WHERE commune = ?', (nom,))
        resultat = cursor.fetchone()

        if resultat and resultat[0] == empreinte_actuelle:
            print("⏭️ Le site n'a pas été modifié depuis le dernier passage. Ignoré (0 appel API).")
            continue

        # --- 3. LE FILTRE PYTHON (Exige au moins 2 mots-clés distincts) ---
        texte_minuscule = texte_brut.lower()
        mots_trouves = [mot for mot in mots_cles if mot in texte_minuscule]

        # S'il y a moins de 2 mots-clés, on écarte tout de suite (0 appel IA)
        if len(mots_trouves) < 2:
            print(f"⏭️ {nom} : Seulement {len(mots_trouves)} mot(s)-clé(s) trouvé(s) ({', '.join(mots_trouves) if mots_trouves else 'aucun'}). Ignoré.")
            cursor.execute('INSERT OR REPLACE INTO historique (commune, empreinte_texte) VALUES (?, ?)', (nom, empreinte_actuelle))
            conn.commit()
            continue

        print(f"✅ {nom} : {len(mots_trouves)} mots-clés détectés ({', '.join(mots_trouves)}). Envoi à l'IA...")

        # --- 4. LE PROMPT IA FORMAT JSON ---
        prompt = f"""
        Tu es un analyste en veille commerciale spécialisé dans la voirie, la signalisation et le mobilier urbain (pour l'entreprise Signature).
        Voici le texte brut extrait du site de la mairie de {nom}.

        Ta mission :
        1. Analyse si le texte contient des informations ou des projets liés à nos métiers (sécurité routière, vitesse, aménagements de voirie, mobilier urbain, signalisation, petits budgets < 100 000 €).
        2. Ignore les actualités sans rapport (sport, culture, cantine).

        Réponds EXCLUSIVEMENT au format JSON strict (sans texte autour, sans markdown ```json si possible), avec la structure suivante :
        {{
            "est_pertinent": true,
            "type_projet": "Titre court du projet (ex: Sécurité Routière - Coussin berlinois)",
            "contexte": "Résumé court du contexte et des besoins",
            "signal_fort": "Détails précis du projet identifié",
            "action_commerciale": "Conseil d'action précis pour le commercial terrain"
        }}
        Si le texte n'a aucun rapport ou est un faux positif, réponds uniquement :
        {{"est_pertinent": false}}

        Texte brut :
        {texte_brut}
        """

        reponse_ia = model.generate_content(prompt)

        # Nettoyage et conversion de la réponse de l'IA en dictionnaire Python (JSON)
        try:
            texte_reponse = reponse_ia.text.replace("```json", "").replace("```", "").strip()
            data_ia = json.loads(texte_reponse)

            if data_ia.get("est_pertinent") == True:
                print(f"🤖 {nom} : OPPORTUNITÉ TROUVÉE !")
                opportunites_structurees.append({
                    "commune": nom,
                    "type_projet": data_ia.get("type_projet", "Projet voirie"),
                    "contexte": data_ia.get("contexte", ""),
                    "signal_fort": data_ia.get("signal_fort", ""),
                    "action_commerciale": data_ia.get("action_commerciale", ""),
                    "url": url
                })
                opportunites_trouvees += 1
            else:
                print(f"⏭️ {nom} : Faux positif ou non pertinent.")

            cursor.execute('INSERT OR REPLACE INTO historique (commune, empreinte_texte) VALUES (?, ?)', (nom, empreinte_actuelle))
            conn.commit()

        except Exception as json_err:
            print(f"⚠️ Erreur de lecture JSON pour {nom} : {json_err}")
            cursor.execute('INSERT OR REPLACE INTO historique (commune, empreinte_texte) VALUES (?, ?)', (nom, empreinte_actuelle))
            conn.commit()

    except Exception as e:
        print(f"❌ Erreur : {e}")

# Fermeture propre de la base de données
conn.close()

# --- 5. ENVOI DU RAPPORT EN CARTES HTML STRUCTURÉES ---
if opportunites_trouvees > 0:
    print(f"\n🚀 Envoi du mail avec {opportunites_trouvees} opportunités structurées...")

    EMAIL_EXPEDITEUR = userdata.get('EMAIL_EXPEDITEUR')
    MOT_DE_PASSE_APP = userdata.get('MDP_APP')
    EMAIL_DESTINATAIRE = userdata.get('EMAIL_DESTINATAIRE')

    msg = EmailMessage()
    msg['Subject'] = f"🚀 Veille Signature : {opportunites_trouvees} opportunité(s) détectée(s)"
    msg['From'] = EMAIL_EXPEDITEUR
    msg['To'] = EMAIL_DESTINATAIRE

    cartes_html = ""
    texte_fallback_brut = ""

    for opp in opportunites_structurees:
        cartes_html += f"""
        <div style="background-color: #ffffff; border: 1px solid #e1e4e8; border-left: 5px solid #0b3c5d; padding: 15px; margin-bottom: 20px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
            <h3 style="margin-top: 0; color: #0b3c5d; font-size: 18px;">📍 {opp['commune'].upper()} — <span style="font-weight: normal; color: #328cc1;">{opp['type_projet']}</span></h3>
            <p style="margin: 8px 0;"><strong>Contexte :</strong> {opp['contexte']}</p>
            <p style="margin: 8px 0;"><strong>Projet / Signal fort :</strong> {opp['signal_fort']}</p>
            <div style="background-color: #f4f7f6; padding: 10px; border-radius: 4px; margin-top: 10px;">
                <p style="margin: 0; color: #1d2731;"><strong>💡 Action commerciale :</strong> {opp['action_commerciale']}</p>
            </div>
            <p style="margin: 12px 0 0 0;"><a href="{opp['url']}" target="_blank" style="color: #328cc1; text-decoration: none; font-weight: bold;">🔗 Accéder au site de la commune →</a></p>
        </div>
        """
        texte_fallback_brut += f"COMMUNE : {opp['commune']}\nProjet : {opp['type_projet']}\nLien : {opp['url']}\n\n"

    msg.set_content(f"Bonjour,\n\nVoici les nouvelles opportunités détectées :\n\n{texte_fallback_brut}")

    html_content = f"""
    <html>
      <body style="font-family: Arial, sans-serif; color: #333; line-height: 1.6; background-color: #f4f7f6; padding: 20px;">
        <div style="max-width: 650px; margin: auto; background: #fdfdfd; padding: 25px; border-radius: 8px;">
          <h2 style="color: #0b3c5d; border-bottom: 2px solid #328cc1; padding-bottom: 10px; margin-top: 0;">
            🚀 Veille Commerciale — Signature ({opportunites_trouvees})
          </h2>
          <p>Bonjour,</p>
          <p>Voici le récapitulatif des opportunités qualifiées par l'intelligence artificielle cette semaine :</p>

          <br>
          {cartes_html}

          <p style="font-size: 12px; color: #888; margin-top: 30px; text-align: center;">
            Rapport automatique généré par votre système de veille. Bon terrain !
          </p>
        </div>
      </body>
    </html>
    """

    msg.add_alternative(html_content, subtype='html')

    try:
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(EMAIL_EXPEDITEUR, MOT_DE_PASSE_APP)
        server.send_message(msg)
        server.quit()
        print("✅ Rapport HTML structuré envoyé avec succès !")
    except Exception as e:
        print(f"❌ Erreur lors de l'envoi de l'email : {e}")
else:
    print("\n😴 Aucune nouveauté cette semaine, pas d'email envoyé.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.32.0 requires pr

/tmp/ipykernel_1498/1994771900.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ 517 mairies chargées avec succès et URLs nettoyées !

🌍 Analyse en cours pour : Bray-Dunes
✅ Bray-Dunes : 2 mots-clés détectés (stationnement, travaux). Envoi à l'IA...
⏭️ Bray-Dunes : Faux positif ou non pertinent.

🌍 Analyse en cours pour : Mouvaux
⏭️ Mouvaux : Seulement 1 mot(s)-clé(s) trouvé(s) (travaux). Ignoré.

🌍 Analyse en cours pour : Avesnelles
⏭️ Avesnelles : Seulement 0 mot(s)-clé(s) trouvé(s) (aucun). Ignoré.

🌍 Analyse en cours pour : Proville
⏭️ Proville : Seulement 0 mot(s)-clé(s) trouvé(s) (aucun). Ignoré.

🌍 Analyse en cours pour : Cobrieux
❌ Erreur : HTTPSConnectionPool(host='mairiecobrieux.fr', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7e84b41f7890>, 'Connection to mairiecobrieux.fr timed out. (connect timeout=10)'))

🌍 Analyse en cours pour : Solesmes
✅ Solesmes : 2 mots-clés détectés (stationnement, panneau). Envoi à l'IA...
⏭️ Solesmes : Faux positif ou non pertinent.

🌍 Analyse en

In [1]:
import requests
import csv

def recuperer_mairies_nord_api_officielle():
    # L'URL de base pour l'export JSON du jeu de données api-lannuaire-administration[cite: 3]
    url = "https://api-lannuaire.service-public.gouv.fr/api/explore/v2.1/catalog/datasets/api-lannuaire-administration/exports/json"

    # Définition des paramètres de la requête selon la documentation
    # - startswith est conseillé pour la recherche par département (on recherche les 2 premiers chiffres du code INSEE)[cite: 5]
    # - pivot LIKE "mairie" permet de rechercher un service local spécifique[cite: 6, 9]
    # - select permet de choisir les champs à retourner, séparés par une virgule[cite: 4]
    params = {
        "where": 'startswith(code_insee_commune,"59") and pivot LIKE "mairie"',
        "select": "nom,site_internet"
    }

    print("Interrogation de l'API Annuaire de l'administration en cours...")

    try:
        # Envoi de la requête
        response = requests.get(url, params=params)
        response.raise_for_status() # Vérifie qu'il n'y a pas d'erreur HTTP

        donnees = response.json()
        mairies_csv = []

        # Traitement des données retournées par l'API
        for mairie in donnees:
            nom_brut = mairie.get("nom", "Inconnu")
            # Nettoyage du nom pour ne garder que la ville
            ville = nom_brut.replace("Mairie - ", "").replace("Mairie et agence postale communale - ", "").strip()

            # Le champ site_internet peut parfois être une liste ou une chaîne, on gère les cas
            site_web = mairie.get("site_internet", "")
            if isinstance(site_web, list) and len(site_web) > 0:
                site_web = site_web[0]

            mairies_csv.append({
                "ville": ville,
                "site_web": site_web if site_web else None
            })

        # Sauvegarde en CSV
        fichier_csv = "mairies_nord_sites.csv"
        with open(fichier_csv, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["ville", "site_web"])
            writer.writeheader()
            writer.writerows(mairies_csv)

        mairies_avec_site = sum(1 for m in mairies_csv if m["site_web"])
        print(f"Terminé ! {len(mairies_csv)} mairies trouvées dans le Nord.")
        print(f"{mairies_avec_site} d'entre elles possèdent un site web renseigné.")
        print(f"Les données ont été sauvegardées dans le fichier : {fichier_csv}")

    except requests.exceptions.RequestException as e:
        print(f"Une erreur s'est produite lors de la connexion à l'API : {e}")

if __name__ == "__main__":
    recuperer_mairies_nord_api_officielle()

Interrogation de l'API Annuaire de l'administration en cours...
Terminé ! 650 mairies trouvées dans le Nord.
525 d'entre elles possèdent un site web renseigné.
Les données ont été sauvegardées dans le fichier : mairies_nord_sites.csv
